# CrashDiag SFT evaluation — independent Kaggle notebook

This notebook starts in a fresh Kaggle GPU session, downloads and hash-verifies the completed SFT adapter plus `grpo_eval.jsonl` from the private `devaanshpa/CrashDiag` Storage Bucket, generates one action for every held-out row, and replays that action against the exact seeded scenario on the authenticated Vultr sandbox. Success is computed only from live sandbox state; no LLM grades another LLM.

Enable **GPU** and **Internet** and attach Kaggle Secrets named `HF_TOKEN` and `CRASHDIAG_SANDBOX_TOKEN`. This is evaluation-only: it does not run SFT or GRPO and does not update model weights.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import subprocess
import sys

REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
SANDBOX_URL = "https://sandbox.devaanshpathak.com"
RUN_ID = os.environ.get("CRASHDIAG_RUN_ID") or "20260719T113724Z-dataset-b26381b116bc"
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT") or "d226578ab4cefadb3c18f2af46b773987d80c7e7"
EVALUATION_STAGE = "sft-eval"
PRECISION = "auto"
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 96
SEED = 42

if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
    raise ValueError("SOURCE_COMMIT must be a full 40-character lowercase Git SHA")
print(f"RUN_ID={RUN_ID}")
print(f"SOURCE_COMMIT={SOURCE_COMMIT}")
print(f"evaluation_stage={EVALUATION_STAGE}")
print(f"sandbox={SANDBOX_URL}")

## Install the exact repository revision

The notebook uses the current notebook workflow but checks the imported CrashDiag code out to the revision recorded in the dataset and SFT manifests.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{REPO_BRANCH}"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-repository directory: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"Git checkout mismatch: expected {SOURCE_COMMIT}, got {CURRENT_COMMIT}")

# TorchAO is optional and CrashDiag does not use it. Some Kaggle images ship
# an old build that makes current PEFT fail while loading ordinary LoRA.
torchao_probe = subprocess.run(
    [sys.executable, "-m", "pip", "show", "torchao"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)
if torchao_probe.returncode == 0:
    print("removing unused preinstalled torchao to avoid a PEFT version conflict")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_source_commit={CURRENT_COMMIT}")

## Load secrets without displaying them

In [ ]:
def required_secret(name: str) -> str:
    existing = os.environ.get(name)
    if existing:
        return existing
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach the Kaggle Secret {name!r} before continuing") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_SANDBOX_TOKEN"] = required_secret("CRASHDIAG_SANDBOX_TOKEN")
os.environ["CRASHDIAG_SANDBOX_URL"] = SANDBOX_URL
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = RUN_ID
print("Kaggle Secrets loaded into the process environment (values hidden).")

## Download and verify the SFT/evaluation handoff

In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader

artifact_config = ArtifactConfig(
    bucket_id=BUCKET_ID,
    run_id=RUN_ID,
    prefix="runs",
    policy="required",
    local_root=REPO_DIR / "artifacts",
    token=os.environ["HF_TOKEN"],
)
uploader = ArtifactUploader(artifact_config)
RUN_DIR = artifact_config.local_run_root
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    uploader.verify_local_run(RUN_DIR, require_complete=False)
    print(f"verified existing recovery directory: {RUN_DIR}")
else:
    uploader.download_run(RUN_DIR, allow_incomplete=True)
    print(f"downloaded and verified: {uploader.remote_uri()}")

SFT_DIR = RUN_DIR / "sft"
DATASET_DIR = RUN_DIR / "datasets"
EVAL_FILE = DATASET_DIR / "grpo_eval.jsonl"
required_handoff = [
    SFT_DIR / "adapter_config.json",
    SFT_DIR / "adapter_model.safetensors",
    SFT_DIR / "manifest.json",
    SFT_DIR / "_SUCCESS.json",
    EVAL_FILE,
    DATASET_DIR / "manifest.json",
    DATASET_DIR / "_SUCCESS.json",
]
missing = [str(path) for path in required_handoff if not path.is_file()]
if missing:
    raise RuntimeError(f"Refusing to evaluate without completed SFT/dataset artifacts: {missing}")
for manifest_path in (SFT_DIR / "manifest.json", DATASET_DIR / "manifest.json"):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    artifact_commit = manifest.get("runtime", {}).get("git_commit")
    if artifact_commit != CURRENT_COMMIT:
        raise RuntimeError(
            f"Artifact/code revision mismatch for {manifest_path.parent.name}: "
            f"artifact={artifact_commit!r}, notebook={CURRENT_COMMIT!r}"
        )
print(f"verified_eval_file={EVAL_FILE}")

## Probe the authenticated Vultr sandbox and reserve the evaluation stage

In [ ]:
from crashdiag.sandbox_apps.http import HttpSandbox

with HttpSandbox(
    SANDBOX_URL,
    api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"],
    timeout=30.0,
) as probe:
    service = probe.service_health()
    observation = probe.observe()
if service.get("status") != "ok":
    raise RuntimeError(f"Vultr sandbox is not healthy: {service}")
if not isinstance(observation, dict):
    raise RuntimeError("Vultr sandbox returned an invalid observation")
uploader.start_stage(
    EVALUATION_STAGE,
    {
        "model": "sft",
        "dataset": "datasets/grpo_eval.jsonl",
        "scoring": "mechanical_fault_resolution",
        "remote_sandbox": True,
    },
)
print("authenticated Vultr preflight passed; evaluation stage reserved")

## Load the signed SFT adapter for deterministic generation

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM, PeftConfig
from transformers import AutoTokenizer, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before loading the model")
if PRECISION == "bf16" or (PRECISION == "auto" and torch.cuda.is_bf16_supported()):
    model_dtype = torch.bfloat16
elif PRECISION in {"auto", "fp16"}:
    model_dtype = torch.float16
else:
    model_dtype = torch.float32

set_seed(SEED)
model = AutoPeftModelForCausalLM.from_pretrained(
    str(SFT_DIR),
    device_map="auto",
    dtype=model_dtype,
)
try:
    tokenizer = AutoTokenizer.from_pretrained(str(SFT_DIR))
except OSError:
    adapter_config = PeftConfig.from_pretrained(str(SFT_DIR))
    tokenizer = AutoTokenizer.from_pretrained(adapter_config.base_model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model.eval()

def generate_completion(messages):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(
        rendered,
        return_tensors="pt",
        add_special_tokens=False,
    )
    encoded = {name: value.to(model.device) for name, value in encoded.items()}
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if TEMPERATURE > 0:
        generation_kwargs["temperature"] = TEMPERATURE
    generated = model.generate(**encoded, **generation_kwargs)
    prompt_length = encoded["input_ids"].shape[-1]
    return tokenizer.decode(generated[0][prompt_length:], skip_special_tokens=True).strip()

print(f"loaded_sft_adapter={SFT_DIR}")
print(f"dtype={model_dtype}, device={model.device}")

## Evaluate every `grpo_eval.jsonl` row mechanically

Each model completion is generated once at temperature zero. The reward function reconstructs the row's exact seeded scenario on Vultr, verifies that the supplied prompt matches that state, executes one bounded action, and checks real health/fault state.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from crashdiag.agents import ACTION_SPACE
from training.common import FAULT_NAMES
from training.grpo import configure_reward_backend, mechanical_reward

rows = [json.loads(line) for line in EVAL_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
if not rows:
    raise RuntimeError("grpo_eval.jsonl is empty")
if any("completion" in row or "answer" in row or "target" in row for row in rows):
    raise RuntimeError("GRPO evaluation rows must remain answer-free")
fault_counts = Counter(str(row.get("fault_name")) for row in rows)
if set(fault_counts) != set(FAULT_NAMES):
    raise RuntimeError(f"Evaluation dataset fault coverage is invalid: {fault_counts}")

configure_reward_backend(
    sandbox_url=SANDBOX_URL,
    api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"],
    timeout=30.0,
)
results = []
for index, row in enumerate(rows, start=1):
    if not isinstance(row.get("prompt"), list) or not row["prompt"]:
        raise RuntimeError(f"row {index} does not contain conversational prompt messages")
    generation_error = None
    try:
        with torch.inference_mode():
            completion = generate_completion(row["prompt"])
    except Exception as exc:
        completion = ""
        generation_error = type(exc).__name__

    extra = {}
    def capture_extra(name, values):
        extra[name] = list(values)

    reward = mechanical_reward(
        completions=[completion],
        fault_name=[row["fault_name"]],
        sample_seed=[row["sample_seed"]],
        prompts=[row["prompt"]],
        log_extra=capture_extra,
    )[0]
    try:
        decoded = json.loads(completion)
        strict_json = (
            isinstance(decoded, dict)
            and decoded.get("action") in ACTION_SPACE
            and isinstance(decoded.get("parameters", {}), dict)
        )
    except (TypeError, ValueError, json.JSONDecodeError):
        strict_json = False
    result = {
        "index": index - 1,
        "fault_name": row["fault_name"],
        "difficulty": row["difficulty"],
        "variation_index": row["variation_index"],
        "sample_seed": row["sample_seed"],
        "completion": completion,
        "action": extra["crashdiag_action"][0],
        "strict_json": strict_json,
        "generation_error": generation_error,
        "backend_error": bool(extra["crashdiag_backend_error"][0]),
        "resolved": bool(extra["crashdiag_resolved"][0]),
        "reward": float(reward),
    }
    results.append(result)
    if index == 1 or index % 8 == 0 or index == len(rows):
        resolved_so_far = sum(item["resolved"] for item in results)
        print(f"evaluated {index}/{len(rows)}; resolved={resolved_so_far}/{index}")

per_fault = {}
for fault_name in FAULT_NAMES:
    matching = [item for item in results if item["fault_name"] == fault_name]
    resolved = sum(item["resolved"] for item in matching)
    per_fault[fault_name] = {
        "difficulty": matching[0]["difficulty"],
        "episodes": len(matching),
        "resolved": resolved,
        "success_rate": resolved / len(matching),
        "backend_errors": sum(item["backend_error"] for item in matching),
        "strict_json_completions": sum(item["strict_json"] for item in matching),
    }
resolved_total = sum(item["resolved"] for item in results)
evaluation = {
    "schema_version": 1,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "run_id": RUN_ID,
    "source_commit": CURRENT_COMMIT,
    "model_stage": "sft",
    "dataset": "datasets/grpo_eval.jsonl",
    "scoring": "mechanical_fault_resolution",
    "generation": {
        "temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "seed": SEED,
    },
    "summary": {
        "total_episodes": len(results),
        "resolved_episodes": resolved_total,
        "success_rate": resolved_total / len(results),
        "backend_errors": sum(item["backend_error"] for item in results),
        "strict_json_completions": sum(item["strict_json"] for item in results),
        "generation_errors": sum(item["generation_error"] is not None for item in results),
    },
    "per_fault": per_fault,
    "results": results,
}
print(json.dumps(evaluation["summary"], indent=2, sort_keys=True))

## Generate graphs, upload signed artifacts, and display results

In [ ]:
from IPython.display import Markdown, SVG, display
from training.reporting import generate_evaluation_report

OUTPUT_DIR = REPO_DIR / "outputs" / EVALUATION_STAGE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_PATH = OUTPUT_DIR / "evaluation.json"
EVALUATION_PATH.write_text(
    json.dumps(evaluation, indent=2, sort_keys=True, allow_nan=False) + "\n",
    encoding="utf-8",
)
report_bundle = generate_evaluation_report(
    EVALUATION_PATH,
    OUTPUT_DIR / "reports",
    title="CrashDiag SFT success on grpo_eval.jsonl",
)
artifact_files = [EVALUATION_PATH, *report_bundle.files]
uploader.upload_files(
    artifact_files,
    EVALUATION_STAGE,
    metadata={
        "model_stage": "sft",
        "dataset": "datasets/grpo_eval.jsonl",
        "summary": evaluation["summary"],
        "scoring": "mechanical_fault_resolution",
        "report": report_bundle.summary,
    },
)

stage_dir = RUN_DIR / EVALUATION_STAGE
manifest_path = stage_dir / "manifest.json"
success_path = stage_dir / "_SUCCESS.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
success = json.loads(success_path.read_text(encoding="utf-8"))
expected_names = {path.name for path in artifact_files}
manifest_names = {entry["path"] for entry in manifest["files"]}
if expected_names != manifest_names:
    raise RuntimeError(f"Evaluation manifest mismatch: expected={expected_names}, actual={manifest_names}")
manifest_sha256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
if success.get("manifest_sha256") != manifest_sha256:
    raise RuntimeError("Evaluation success marker does not match its manifest")

display(Markdown(report_bundle.markdown_path.read_text(encoding="utf-8")))
for chart in report_bundle.charts:
    display(SVG(filename=str(chart)))
print(f"signed evaluation artifacts: {uploader.remote_uri(EVALUATION_STAGE)}")
print("Run-level completion is intentionally unchanged; this notebook evaluates SFT only.")